# QisDebug demo

Attach full lifecycle visibility to ordinary Qiskit code with one activation call.
Everything below the activation cell is plain Qiskit — no QisDebug-specific calls
until we read the timeline back at the end.

In [1]:
import qisdebug

report = qisdebug.activate()
report.layers

{'construction': True, 'execution': True, 'transpile': True}

## Ordinary Qiskit code

In [2]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

qc = QuantumCircuit(3, 3, name="ghz")
qc.h(0)
qc.cx(0, 1)
qc.cx(1, 2)
qc.measure(range(3), range(3))

backend = AerSimulator()
transpiled = transpile(qc, basis_gates=["cx", "u"], optimization_level=1)
counts = backend.run(transpiled, shots=1000).result().get_counts()
counts

{'000': 517, '111': 483}

## What QisDebug saw

Construction, transpilation and execution all landed on one timeline for `qc`,
even though the circuit that actually ran was the transpiled copy.

In [3]:
timeline = qisdebug.timeline(qc)
print(timeline.summary())

Timeline 'ghz' (id=e159bc8dd21c, 3 qubits, 3 clbits, 33 events)
  gate_added             6
  transpile_started      1
  transpile_pass         19
  transpile_finished     1
  execution_submitted    1
  state_snapshot         4
  execution_finished     1


In [4]:
for event in timeline.events:
    print(f"[{event.seq:>3}] {event.type.value:<22} {event.describe()}")

[  3] gate_added             h q[0]
[  4] gate_added             cx q[0, 1]
[  5] gate_added             cx q[1, 2]
[  6] gate_added             measure q[0] -> c[0]
[  7] gate_added             measure q[1] -> c[1]
[  8] gate_added             measure q[2] -> c[2]
[  9] transpile_started      transpile started (StagedPassManager, depth=4, size=6)
[ 10] transpile_pass         pass* ContainsInstruction (depth=4, size=6)
[ 11] transpile_pass         pass  InverseCancellation (depth=4, size=6)
[ 12] transpile_pass         pass  ContractIdleWiresInControlFlow (depth=4, size=6)
[ 13] transpile_pass         pass  UnitarySynthesis (depth=4, size=6)
[ 14] transpile_pass         pass  HighLevelSynthesis (depth=4, size=6)
[ 15] transpile_pass         pass* BasisTranslator (depth=4, size=6)
[ 16] transpile_pass         pass  Size (depth=4, size=6)
[ 17] transpile_pass         pass  Depth (depth=4, size=6)
[ 18] transpile_pass         pass  FixedPoint (depth=4, size=6)
[ 19] transpile_pass        

## State evolution

The snapshots are real simulator statevectors captured during the run you
already made, then stripped from the result object you received.

In [5]:
for snapshot in timeline.snapshots:
    probabilities = {
        format(i, f"0{snapshot.num_qubits}b"): round(p, 3)
        for i, p in enumerate(snapshot.probabilities())
        if p > 1e-9
    }
    print(f"step {snapshot.step} ({snapshot.gate or 'initial':<7}) {probabilities}")

step 0 (initial) {'000': 1.0}
step 1 (u      ) {'000': 0.5, '001': 0.5}
step 2 (cx     ) {'000': 0.5, '011': 0.5}
step 3 (cx     ) {'000': 0.5, '111': 0.5}


## Interactive timeline widget

Requires the viz extra: `pip install "qisdebug[viz]"` (ipywidgets). A slider
walks every event, drawing the circuit as it existed at that moment and the
captured statevectors.

In [6]:
timeline.show()

In [7]:
qisdebug.deactivate()